# Test-set ROC / PR — mirrors `plot_bench_auroc_auprc.ipynb`

**Input**: `results/test/run_seed*/test_log_full_<STAMP>.jsonl` (batched `probs`, `labels`, `sample_ids`).

**Pooling** defaults to **`protein_max`**: protein score is **max residue prob**; protein label is **1 if any** residue positive (benchmark-style enzyme-vs-non enzyme fold labeling).

Alt: set **`AGGREGATION = "residue"`** for residue-level ROC/PR.

Edit **`STAMP`** / **`USE_DELETE_JSONL`**. Saves PNG beside this notebook.

In [ ]:
%matplotlib inline

import json
import re
from pathlib import Path
from typing import List, Tuple

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)


In [ ]:
def normalize_json_nan_line(line: str) -> str:
    line = re.sub(r":\s*-?Infinity\b", ": null", line)
    return re.sub(r":\s*NaN\b", ": null", line)


def stream_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            s = line.strip()
            if not s:
                continue
            yield json.loads(normalize_json_nan_line(s))


def residue_level_arrays(jsonl_path: Path):
    y_true_list, y_score_list = [], []
    for row in stream_jsonl(jsonl_path):
        for p, lbl in zip(row["probs"], row["labels"]):
            y_score_list.append(float(p))
            y_true_list.append(float(lbl))
    yt = np.asarray(y_true_list, dtype=np.float64)
    ys = np.asarray(y_score_list, dtype=np.float64)
    return yt, ys


def protein_max_arrays(jsonl_path: Path):
    """Per accession: score = max prob; label positive if any residue is positive."""
    max_p = {}
    any_pos = {}
    for row in stream_jsonl(jsonl_path):
        for sid, p, lbl in zip(row["sample_ids"], row["probs"], row["labels"]):
            head = sid.split("|", 1)[0]
            if not head.endswith(".pdb"):
                continue
            acc = head[:-4]
            p_f = float(p)
            lbl_f = float(lbl)
            if acc not in max_p:
                max_p[acc] = p_f
                any_pos[acc] = lbl_f >= 0.5
            else:
                if p_f > max_p[acc]:
                    max_p[acc] = p_f
                if lbl_f >= 0.5:
                    any_pos[acc] = True
    accs = sorted(max_p.keys())
    y_true = np.array([1.0 if any_pos[a] else 0.0 for a in accs])
    y_score = np.array([max_p[a] for a in accs], dtype=np.float64)
    return y_true, y_score


def build_arrays(kind: str, jsonl_path: Path):
    if kind == "residue":
        return residue_level_arrays(jsonl_path)
    if kind == "protein_max":
        return protein_max_arrays(jsonl_path)
    raise ValueError("AGGREGATION must be 'residue' or 'protein_max'")


def plot_test_roc_prc_overlay(series, *, title: str, save_path=None):
    """Same ROC | PR overlay pattern as plot_bench_auroc_auprc notebook."""
    colors = ["#2563EB", "#059669", "#D97706", "#7C3AED", "#DC2626"]
    fig, (ax_roc, ax_pr) = plt.subplots(
        1,
        2,
        figsize=(12, 5.2),
        dpi=150,
        gridspec_kw={"width_ratios": [1, 1], "wspace": 0.28},
    )

    prevalence = float(np.mean(series[0][1]))

    aurocs_list = []
    auprcs_list = []

    for i, (seed_label, y_true, y_score) in enumerate(series):
        y_true_i = np.asarray(y_true).astype(np.int32).ravel()
        y_score_i = np.asarray(y_score, dtype=np.float64).ravel()
        uniq = np.unique(y_true_i)
        if uniq.size < 2:
            print(f"[warn] seed {seed_label}: single class {uniq.tolist()}, skip curve")
            continue

        auroc = roc_auc_score(y_true_i, y_score_i)
        auprc = average_precision_score(y_true_i, y_score_i)
        aurocs_list.append(float(auroc))
        auprcs_list.append(float(auprc))

        fpr, tpr, _ = roc_curve(y_true_i, y_score_i)
        prec, rec, _ = precision_recall_curve(y_true_i, y_score_i)
        c = colors[i % len(colors)]

        ax_roc.plot(
            fpr,
            tpr,
            color=c,
            lw=2,
            label=f"seed {seed_label} (AUROC = {auroc:.4f})",
        )
        ax_pr.plot(
            rec,
            prec,
            color=c,
            lw=2,
            label=f"seed {seed_label} (AUPRC = {auprc:.4f})",
        )

    ax_roc.plot([0, 1], [0, 1], "k--", lw=1, alpha=0.45, label="chance")
    ax_roc.set_xlabel("False positive rate")
    ax_roc.set_ylabel("True positive rate")
    ax_roc.set_title("ROC")
    ax_roc.set_xlim(-0.02, 1.02)
    ax_roc.set_ylim(-0.02, 1.02)
    ax_roc.legend(loc="lower right", fontsize=8, framealpha=0.92)
    ax_roc.grid(True, linestyle=":", alpha=0.45)

    ax_pr.axhline(
        prevalence,
        color="k",
        linestyle="--",
        lw=1,
        alpha=0.45,
        label=f"random baseline (prevalence = {prevalence:.4f})",
    )
    ax_pr.set_xlabel("Recall")
    ax_pr.set_ylabel("Precision")
    ax_pr.set_title("PRC")
    ax_pr.set_xlim(-0.02, 1.02)
    ax_pr.set_ylim(-0.02, 1.02)
    ax_pr.legend(loc="lower left", fontsize=8, framealpha=0.92)
    ax_pr.grid(True, linestyle=":", alpha=0.45)

    if aurocs_list:
        au = np.array(aurocs_list, dtype=np.float64)
        ap = np.array(auprcs_list, dtype=np.float64)
        n_seeds = len(au)
        std_au = float(au.std(ddof=1)) if n_seeds > 1 else 0.0
        std_ap = float(ap.std(ddof=1)) if n_seeds > 1 else 0.0
        stats_line = (
            f"mean ± std  AUROC: {float(au.mean()):.4f} ± {std_au:.4f}    "
            f"AUPRC: {float(ap.mean()):.4f} ± {std_ap:.4f}    "
            f"(n_seeds = {n_seeds})"
        )
        fig.text(0.5, 0.965, stats_line, ha="center", fontsize=10, transform=fig.transFigure)

    fig.suptitle(title, fontsize=11, y=1.02)
    fig.tight_layout(rect=[0.0, 0.0, 1.0, 0.88])
    if save_path:
        fig.savefig(save_path, bbox_inches="tight", pad_inches=0.25)
    return fig, (ax_roc, ax_pr)


def metrics_summary(series):
    rows = []
    for seed_label, y_true, y_score in series:
        y_true_i = np.asarray(y_true).astype(np.int32).ravel()
        y_score_i = np.asarray(y_score, dtype=np.float64).ravel()
        uniq = np.unique(y_true_i)
        if uniq.size < 2:
            print(f"seed={seed_label}  skip (single-class {uniq.tolist()})")
            continue
        auroc = float(roc_auc_score(y_true_i, y_score_i))
        auprc = float(average_precision_score(y_true_i, y_score_i))
        n_pos = int(np.sum(y_true_i >= 0.5))
        n_tot = int(y_true_i.size)
        rows.append(
            dict(
                seed=str(seed_label),
                n_total=n_tot,
                n_pos=n_pos,
                AUROC=auroc,
                AUPRC=auprc,
            )
        )
        print(
            f"seed={seed_label}  n={n_tot}  pos={n_pos}  "
            f"AUROC={auroc:.6f}  AUPRC={auprc:.6f}"
        )
    if not rows:
        return
    au = np.array([r["AUROC"] for r in rows], dtype=np.float64)
    ap = np.array([r["AUPRC"] for r in rows], dtype=np.float64)
    if au.size >= 2:
        print(f"\nmean ± std  AUROC: {au.mean():.6f} ± {au.std(ddof=1):.6f}")
        print(f"mean ± std  AUPRC: {ap.mean():.6f} ± {ap.std(ddof=1):.6f}")
    try:
        from pandas import DataFrame

        display(DataFrame(rows))
    except Exception:
        pass


In [ ]:
TEST_ROOT = Path("/data/data3/conglab/s441865/code/Catalytic_sites_setA_3dcnn/results/test")
# Default timestamps; override per seed in STAMP_OVERRIDE when folders disagree.
STAMP_DEFAULT = "20260424_201416"
USE_DELETE_JSONL = False
AGGREGATION = "protein_max"  # or "residue"

STAMP_OVERRIDE = {"2026": "20260502_020921"}

def resolved_jsonl(run_seed_folder: Path, stamp: str):
    names = sorted(
        fn
        for fn in run_seed_folder.iterdir()
        if fn.is_file()
        and fn.name.startswith("test_log_full_")
        and fn.name.endswith(".jsonl")
        and ".delete.jsonl" not in fn.name
    )
    if USE_DELETE_JSONL:
        names = sorted(
            fn
            for fn in run_seed_folder.iterdir()
            if fn.is_file()
            and fn.name.startswith("test_log_full_")
            and fn.name.endswith(".delete.jsonl")
        )

    if not names:
        raise FileNotFoundError(f"no matching test_log_full_* in {run_seed_folder}")

    matched = [x for x in names if stamp in x.name]
    if matched:
        return matched[-1]
    raise FileNotFoundError(
        f"no file containing stamp '{stamp}' in {run_seed_folder}; have {names!r}"
    )


SEED_SPECS = [
    ("42", TEST_ROOT / "run_seed42"),
    ("123", TEST_ROOT / "run_seed123"),
    ("2026", TEST_ROOT / "run_seed2026"),
]


def load_series(seed_specs):
    out = []
    for label, folder in seed_specs:
        stamp = STAMP_OVERRIDE.get(str(label), STAMP_DEFAULT)
        path = resolved_jsonl(folder, stamp)
        yt, ys = build_arrays(AGGREGATION, path)
        print(f"seed={label}  file={path.name}  n_points={yt.size}")
        out.append((label, yt, ys))
    return out


SERIES = load_series(SEED_SPECS)


def pooled_label(series):
    if AGGREGATION == "protein_max":
        return "protein-level max(prob)"
    return "residue-level"


POOL_TAG = pooled_label(SERIES)
FIG_TITLE = f"Test set — ROC & PR ({POOL_TAG}; delete={USE_DELETE_JSONL})"
SAVE_PNG = Path("/data/data3/conglab/s441865/code/Catalytic_sites_setA_3dcnn/test_roc_prc_3seeds_overlay.png")


In [ ]:
metrics_summary(SERIES)


In [ ]:
fig, _ = plot_test_roc_prc_overlay(SERIES, title=FIG_TITLE, save_path=str(SAVE_PNG))
plt.show()
